# Compile and Run the Ray Data RAG Pipeline

This notebook runs from an **in-cluster Workbench**. It clones the Alauda `pipelines-components` mirror, compiles `parse_and_chunk` and the experimental `rag_multistep_pipeline`, and uploads the YAML to Kubeflow Pipelines.

How-to page: [Run Ray Data Pipelines from pipelines-components](./run-ray-data-pipelines.mdx).

Ray Data executes **inside** the `parse_and_chunk` RayJob, not in this kernel. This notebook only compiles and submits.


## Parameters

Set `PIPELINES_COMPONENTS_REPO` to the mirror URL for your environment. `RAY_IMAGE` must already contain Ray and Docling and be pullable in the cluster. Do not keep the upstream `quay.io/rhoai-szaher/docling-ray:latest` default.


In [ ]:
import os
from pathlib import Path

NAMESPACE = os.environ.get("RAY_DATA_NAMESPACE", "my-pipeline-namespace")
PIPELINES_COMPONENTS_REPO = os.environ.get(
    "PIPELINES_COMPONENTS_REPO",
    "https://github.com/alauda/pipelines-components.git",
)
REPO_DIR = Path(os.environ.get("PIPELINES_COMPONENTS_DIR", "pipelines-components"))
RAY_IMAGE = os.environ.get("RAY_IMAGE", "<registry>/mlops/docling-ray:<tag>")
PVC_NAME = os.environ.get("RAY_DATA_PVC", "data-pvc")
S3_ENDPOINT = os.environ.get(
    "RAY_DATA_S3_ENDPOINT", "http://seaweedfs.mlops-demo-e2e:8333"
)
S3_BUCKET = os.environ.get("RAY_DATA_S3_BUCKET", "rag-chunks")
S3_SECRET_NAME = os.environ.get("RAY_DATA_S3_SECRET", "minio-secret")
KFP_HOST = os.environ.get("KFP_HOST", "")

print("NAMESPACE", NAMESPACE)
print("RAY_IMAGE", RAY_IMAGE)
print("S3_ENDPOINT", S3_ENDPOINT)
print("S3_BUCKET", S3_BUCKET)


## Install the KFP SDK and the mirror package

Pin `kfp` to 2.16.1 to match Alauda AI. `--no-deps` on the editable install avoids pulling a different SDK.


In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-U", "pip"]
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "kfp==2.16.1", "kfp-kubernetes==2.16.1"]
)

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", PIPELINES_COMPONENTS_REPO, str(REPO_DIR)])

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR)]
)
print("installed", REPO_DIR.resolve())


## Compile `parse_and_chunk` and the RAG pipeline

Both YAML files land in the current working directory. Upload them from the KFP UI, or set `KFP_HOST` and run the submit cell.


In [ ]:
from kfp import compiler
from kfp_components.components.data_processing.parse_and_chunk import parse_and_chunk
from kfp_components.pipelines.data_processing.ray_data.pdf_documents_processing_rag_pipeline import (
    rag_multistep_pipeline,
)

compiler.Compiler().compile(parse_and_chunk, package_path="parse_and_chunk.yaml")
compiler.Compiler().compile(
    rag_multistep_pipeline, package_path="rag_multistep_pipeline.yaml"
)
print("wrote parse_and_chunk.yaml", Path("parse_and_chunk.yaml").stat().st_size, "bytes")
print("wrote rag_multistep_pipeline.yaml", Path("rag_multistep_pipeline.yaml").stat().st_size, "bytes")


## Optional: submit `parse_and_chunk`

Requires `KFP_HOST` (in-cluster Kubeflow Pipelines API). The run uses the PVC, Secret, and `RAY_IMAGE` from the parameters cell. Confirm PDFs exist under `input/pdfs/` on `data-pvc` first.


In [ ]:
if not KFP_HOST:
    print("KFP_HOST is empty; upload parse_and_chunk.yaml from the KFP UI instead.")
else:
    from kfp.client import Client

    client = Client(host=KFP_HOST)
    run = client.create_run_from_pipeline_package(
        "parse_and_chunk.yaml",
        arguments={
            "pvc_name": PVC_NAME,
            "pvc_mount_path": "/mnt/data",
            "input_path": "input/pdfs",
            "ray_image": RAY_IMAGE,
            "namespace": NAMESPACE,
            "s3_endpoint": S3_ENDPOINT,
            "s3_bucket": S3_BUCKET,
            "s3_prefix": "chunks",
            "s3_secret_name": S3_SECRET_NAME,
            "num_files": 0,
            "bypass_kueue": False,
        },
        experiment_name="ray-data-parse-and-chunk",
        namespace=NAMESPACE,
    )
    print(run.run_id)


## Next steps

- Watch the run graph, then `kubectl -n "$NAMESPACE" get rayjobs` for the `docling-chunk-*` job.
- List `s3://<bucket>/chunks/` for one JSONL per PDF.
- Full RAG DAG: [Run Ray Data Pipelines from pipelines-components](./run-ray-data-pipelines.mdx#full-pipeline).
- Interactive Docling: [Prepare Fine-Tuning Data with Docling and SDG Hub](../../train/guides/prepare-finetune-data-with-docling-and-sdg.mdx).
